# Lab 12.3: Concept Bottlenecks, Interventions and Leakage

Chapter 12, Concepts Inside Networks. Run-only notebook, with one optional edit at the end.

A concept bottleneck model (Koh et al., 2020) first predicts human-named concepts, then predicts the label from those concepts only. Its promise: an expert can read the concepts and correct them. Here you build three bottleneck models (independent, sequential, joint) and a standard network on synthetic inspection records, then test two claims: that correcting concepts at test time helps, and that the concepts carry nothing beyond themselves (they may not: Mahinpei et al., 2021, call this leakage).

The data are synthetic and say so: 30 sensor-like measurements per part, generated from four binary concepts (scratch, dent, discoloration, misalignment) and one factor that no one labeled (think of it as lighting or surface finish). A part is defective if it has a scratch or a dent, or discoloration together with misalignment when the unlabeled factor is above 0.4, or whenever the unlabeled factor is above 0.85.

In [1]:
import numpy as np, torch, torch.nn as nn
from sklearn.ensemble import GradientBoostingRegressor
torch.manual_seed(0); torch.set_num_threads(4)
rng = np.random.default_rng(4)
names = ["scratch", "dent", "discoloration", "misalignment"]

def inspection_data(n, rng):
    C = (rng.random((n, 4)) < [0.25, 0.2, 0.4, 0.4]).astype(np.float32)
    h = rng.random(n).astype(np.float32)                         # the unlabeled factor
    W = np.random.default_rng(99).normal(size=(5, 30)).astype(np.float32)
    X = np.c_[C, 3 * h] @ W + rng.normal(0, 1.6, (n, 30)).astype(np.float32)
    y = ((C[:, 0] + C[:, 1] > 0) | ((C[:, 2] * C[:, 3] > 0) & (h > 0.4)) | (h > 0.85)).astype(np.float32)
    return X, C, y, h

Xtr, Ctr, ytr, htr = inspection_data(6000, rng); Xte, Cte, yte, hte = inspection_data(3000, rng)
print("training parts:", len(Xtr), " test parts:", len(Xte), " defect rate: %.2f" % ytr.mean())

training parts: 6000  test parts: 3000  defect rate: 0.55


## 1. Train four models (about 15 seconds)

In [2]:
T = torch.tensor
mlp = lambda i, o: nn.Sequential(nn.Linear(i, 32), nn.ReLU(), nn.Linear(32, o))
bce = nn.functional.binary_cross_entropy_with_logits
X, C, Y = T(Xtr), T(Ctr), T(ytr)

def fit(params, loss_fn, epochs=60):
    opt = torch.optim.Adam(params, 3e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for b in range(0, len(X), 128):
            j = perm[b:b+128]; opt.zero_grad(); loss_fn(j).backward(); opt.step()

g_ind = mlp(30, 4); fit(g_ind.parameters(), lambda j: bce(g_ind(X[j]), C[j]))           # x -> concepts
f_ind = mlp(4, 1);  fit(f_ind.parameters(), lambda j: bce(f_ind(C[j])[:, 0], Y[j]))     # true concepts -> y
with torch.no_grad(): P = torch.sigmoid(g_ind(X))
f_seq = mlp(4, 1);  fit(f_seq.parameters(), lambda j: bce(f_seq(P[j])[:, 0], Y[j]))     # predicted concepts -> y
g_j, f_j = mlp(30, 4), mlp(4, 1)                                                       # joint, concept weight 0.01
fit(list(g_j.parameters()) + list(f_j.parameters()),
    lambda j: bce(f_j(torch.sigmoid(g_j(X[j])))[:, 0], Y[j]) + 0.01 * bce(g_j(X[j]), C[j]))
std = mlp(30, 1); fit(std.parameters(), lambda j: bce(std(X[j])[:, 0], Y[j]))           # no concepts
print("done")

done


## 2. Accuracy, and test-time intervention

In [3]:
Xt = T(Xte)
with torch.no_grad():
    print("standard network accuracy: %.3f" % ((std(Xt)[:, 0] > 0).numpy() == yte).mean())
    print("label from the TRUE concepts (ceiling of an honest bottleneck): %.3f" % ((f_ind(T(Cte))[:, 0] > 0).numpy() == yte).mean())
    for name, g, f in (("independent", g_ind, f_ind), ("sequential", g_ind, f_seq), ("joint", g_j, f_j)):
        Pt = torch.sigmoid(g(Xt)).numpy(); row = []
        for k in range(5):                              # an inspector supplies the first k concepts
            Q = Pt.copy(); Q[:, :k] = Cte[:, :k]
            row.append(((f(T(Q))[:, 0] > 0).numpy() == yte).mean())
        print("%-12s concept accuracy %.3f | label accuracy after correcting 0..4 concepts: %s"
              % (name, ((Pt > 0.5) == Cte).mean(), np.round(row, 3)))

standard network accuracy: 0.836
label from the TRUE concepts (ceiling of an honest bottleneck): 0.889
independent  concept accuracy 0.928 | label accuracy after correcting 0..4 concepts: [0.791 0.852 0.883 0.886 0.889]
sequential   concept accuracy 0.928 | label accuracy after correcting 0..4 concepts: [0.795 0.851 0.879 0.886 0.889]
joint        concept accuracy 0.690 | label accuracy after correcting 0..4 concepts: [0.832 0.825 0.853 0.71  0.626]


**Checkpoint 1.** The standard network scores about 0.84. The independent and sequential bottlenecks start lower (about 0.79 to 0.80) and climb to about 0.89 as an inspector corrects concepts. The joint model starts higher (about 0.83), but its concept accuracy is only about 0.69, and correcting concepts makes it *worse*, down to about 0.63.

## 3. Leakage: what else is inside the concept scores?

If the concept scores contained only the concepts, nothing could recover the unlabeled factor from them. We fit a regression from each set of concept scores to the unlabeled factor and report R squared on the test set.

In [4]:
with torch.no_grad():
    reps = {"true concepts": (Ctr, Cte),
            "independent soft scores": (P.numpy(), torch.sigmoid(g_ind(Xt)).numpy()),
            "joint soft scores": (torch.sigmoid(g_j(X)).numpy(), torch.sigmoid(g_j(Xt)).numpy())}
for name, (A, B) in reps.items():
    pr = GradientBoostingRegressor(random_state=0).fit(A, htr).predict(B)
    print("%-24s R squared for the unlabeled factor: %.2f" % (name, 1 - ((pr - hte) ** 2).sum() / ((hte - hte.mean()) ** 2).sum()))

true concepts            R squared for the unlabeled factor: -0.00


independent soft scores  R squared for the unlabeled factor: 0.08


joint soft scores        R squared for the unlabeled factor: 0.62


**Checkpoint 2.** About 0.00 for the true concepts, about 0.08 for the independent scores and about 0.6 for the joint scores. The joint model's "discoloration" and "misalignment" scores are partly a code for the unlabeled factor. That explains both its higher starting accuracy and its collapse under intervention: replacing a score with the inspector's 0 or 1 erases the hidden code.

## Optional edit

In the joint model, change the concept weight from `0.01` to `1.0` and rerun cells 3 to 5. Predict first: does leakage fall? Does intervention help again? What happens to accuracy before intervention?

## Questions

1. A vendor shows that its bottleneck model is as accurate as a black box and "explains every decision with four inspection concepts." What two numbers from this lab would you ask for before believing the explanation?
2. Why does the independent model start below the standard network, and why can it overtake it after interventions?
3. The unlabeled factor matters for the label. Would adding it as a fifth concept be better than letting it leak? What would you need to label it?